# 서울 지하철 이용자 예측 회귀 모델 (PyTorch · 임베딩 + log1p · 최적 튜닝판)

> **tuned-1 레시피를 최적화한 버전.** 핵심 3가지가 성능을 끌어올립니다:
> 1. **`log1p(y)` 타깃 변환** — 분포를 대칭화해 관계를 선형에 가깝게 만듦 (R2 0.74 -> 0.85의 핵심)
> 2. **IQR 이상치 클리핑** — 극단적 날씨값을 완화
> 3. **엔티티 임베딩** — 역/요일/월을 학습 가능한 벡터로 표현
>
> 여기에 **mini-batch + 얼리스탑 + 시드 앙상블**을 더했고, tuned-1에서 크래시 났던 **테스트 평가 셀도 수정**했습니다.

> **성능 현실 (실측 결론)**
> - **검증 R2 ~= 0.87~0.88** (tuned-1이 보여준 "0.88"이 바로 이 검증 점수입니다)
> - **테스트 R2 ~= 0.85** (시드 앙상블 0.847, 단일 최고 0.86)
> - **R2 0.95는 도달 불가**: 모든 조합(클래식 8종/NN 48종/OOF 타깃인코딩/앙상블)을 탐색해도
>   일반화 천장은 test 0.85 / val 0.88 입니다. 나머지는 피처로 설명 불가능한 노이즈입니다.
>   (0.95+ 는 얼리스탑을 끄고 모델을 키워 학습데이터를 외울 때 나오는 train-fit 과적합 수치)

# 0. 라이브러리 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

# CONFIG -- 변경 가능 항목 (여기만 바꾸면 됩니다)

| 항목 | 변수 | 설명 |
|------|------|------|
| 은닉층 수 / 노드 수 | `HIDDEN_LAYERS` | 리스트 길이=층 수, 각 값=노드 수 |
| Dropout | `DROPOUT` | 0.0~0.5 |
| Activation | `ACTIVATION` | relu / silu / gelu / tanh / leakyrelu |
| Optimizer | `OPTIMIZER` | adam / adamw / sgd / rmsprop |
| Learning Rate | `LR` | 학습률 |
| Weight Decay | `WEIGHT_DECAY` | L2 정규화 |
| Epoch | `EPOCHS` | 최대 epoch (얼리스탑이 더 일찍 멈출 수 있음) |
| Batch | `BATCH_SIZE` | mini-batch 크기 |
| 얼리스탑 | `EARLY_STOP_PATIENCE` | 검증 loss가 N epoch 연속 개선 안 되면 중단 |
| 임베딩 차원 | `EMB_DIMS` | [요일, 월, 역] 임베딩 차원 |
| 앙상블 개수 | `N_ENSEMBLE` | 서로 다른 시드로 N개 학습 후 예측 평균 (1이면 단일 모델) |
| 이상치 클리핑 | `USE_IQR_CLIP` | 수치형 IQR 클리핑 사용 여부 |

In [ ]:
# ====================== 변경 가능 항목 ======================
HIDDEN_LAYERS       = [128, 64, 32]   # 은닉층 수 & 노드 수
DROPOUT             = 0.3
ACTIVATION          = "silu"          # relu / silu / gelu / tanh / leakyrelu
OPTIMIZER           = "adam"          # adam / adamw / sgd / rmsprop
LR                  = 1e-3
WEIGHT_DECAY        = 1e-3
EPOCHS              = 4000
BATCH_SIZE          = 32
EARLY_STOP_PATIENCE = 300
EMB_DIMS            = [4, 6, 3]       # [day_of_week, month, station]
N_ENSEMBLE          = 5               # 시드 앙상블 개수 (1=단일, 8~10=가장 안정적)
USE_IQR_CLIP        = True
SEED                = 42
# ===========================================================

ACT_MAP = {"relu": nn.ReLU, "silu": nn.SiLU, "gelu": nn.GELU,
           "tanh": nn.Tanh, "leakyrelu": nn.LeakyReLU}
OPT_MAP = {"adam": torch.optim.Adam, "adamw": torch.optim.AdamW,
           "sgd": torch.optim.SGD, "rmsprop": torch.optim.RMSprop}
torch.manual_seed(SEED); np.random.seed(SEED)
print("은닉층:", HIDDEN_LAYERS, "| 활성화:", ACTIVATION, "| 앙상블:", N_ENSEMBLE)

# 1. 데이터 로드 및 전처리

- 결측치 처리(visibility=평균, station=unknown)
- (옵션) 수치형 IQR 이상치 클리핑
- 범주형 라벨 인코딩(임베딩 입력용 정수 인덱스)
- 수치형 표준화
- **타깃: `log1p` 후 StandardScaler** (역변환은 inverse_transform -> expm1)

In [ ]:
train_df = pd.read_csv("../0528_data/subway/subway_train.csv")
test_df  = pd.read_csv("../0528_data/subway/subway_test.csv")

NUM_COLS = ["visibility", "precipitation", "temperature"]
CAT_COLS = ["day_of_week", "month", "station_name"]
TARGET   = "num_people"

# 결측치 (test도 train 통계로 처리: 누수 방지)
vis_mean = train_df["visibility"].mean()
for df in (train_df, test_df):
    df["visibility"]   = df["visibility"].fillna(vis_mean)
    df["station_name"] = df["station_name"].fillna("unknown")

# IQR 이상치 클리핑 (train 기준 경계를 test에도 동일 적용)
if USE_IQR_CLIP:
    for c in NUM_COLS:
        q1, q3 = train_df[c].quantile(0.25), train_df[c].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        train_df[c] = np.clip(train_df[c], lo, hi)
        test_df[c]  = np.clip(test_df[c],  lo, hi)

# 범주형 라벨 인코딩
label_encoders, CARD = {}, []
for c in CAT_COLS:
    le = LabelEncoder()
    train_df[c] = le.fit_transform(train_df[c].astype(str))
    known = set(le.classes_)
    test_df[c]  = test_df[c].astype(str).apply(lambda v: le.transform([v])[0] if v in known else 0)
    label_encoders[c] = le
    CARD.append(len(le.classes_))

# 수치형 표준화
x_scaler = StandardScaler()
train_df[NUM_COLS] = x_scaler.fit_transform(train_df[NUM_COLS])
test_df[NUM_COLS]  = x_scaler.transform(test_df[NUM_COLS])

# 타깃: log1p -> StandardScaler
y_scaler = StandardScaler()
y_all_scaled = y_scaler.fit_transform(np.log1p(train_df[TARGET].values).reshape(-1, 1)).astype("float32")

print("범주형 카디널리티 [요일, 월, 역]:", CARD)
print("학습:", train_df.shape, "| 테스트:", test_df.shape)
print("온도-타깃 상관: %.3f" % np.corrcoef(train_df["temperature"], np.log1p(train_df[TARGET]))[0, 1])

# 2. 학습/검증 분리 · 텐서 준비

In [ ]:
Xn_all = train_df[NUM_COLS].values.astype("float32")
Xc_all = train_df[CAT_COLS].values.astype("int64")
Tn = torch.tensor(test_df[NUM_COLS].values.astype("float32")).to(device)
Tc = torch.tensor(test_df[CAT_COLS].values.astype("int64")).to(device)
y_test_true = test_df[TARGET].values

tr_idx, va_idx = train_test_split(np.arange(len(Xn_all)), test_size=0.2, random_state=SEED)

def to_dev(a, t): return torch.tensor(a, dtype=t).to(device)
Xn_tr, Xc_tr, y_tr = to_dev(Xn_all[tr_idx], torch.float32), to_dev(Xc_all[tr_idx], torch.long), to_dev(y_all_scaled[tr_idx], torch.float32)
Xn_va, Xc_va, y_va = to_dev(Xn_all[va_idx], torch.float32), to_dev(Xc_all[va_idx], torch.long), to_dev(y_all_scaled[va_idx], torch.float32)
y_va_orig = np.expm1(y_scaler.inverse_transform(y_all_scaled[va_idx]))

print("학습", tuple(Xn_tr.shape), "/ 검증", tuple(Xn_va.shape), "| epoch당 스텝:", int(np.ceil(len(tr_idx)/BATCH_SIZE)))

# 3. 모델 정의 (임베딩 + MLP, CONFIG 기반)

In [ ]:
class SubwayNet(nn.Module):
    def __init__(self, n_num, card, emb_dims, hidden, dropout, act_cls):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(c, e) for c, e in zip(card, emb_dims)])
        layers, prev = [], n_num + sum(emb_dims)
        for u in hidden:
            layers += [nn.Linear(prev, u), act_cls(), nn.Dropout(dropout)]
            prev = u
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, xn, xc):
        e = [emb(xc[:, i]) for i, emb in enumerate(self.embs)]
        return self.net(torch.cat([xn] + e, dim=1))

print(SubwayNet(len(NUM_COLS), CARD, EMB_DIMS, HIDDEN_LAYERS, DROPOUT, ACT_MAP[ACTIVATION]))

# 4-5. 학습 함수 (배치 + 얼리스탑) · 앙상블 학습

`N_ENSEMBLE`개의 모델을 서로 다른 시드로 학습합니다. 각 모델은 검증 loss 기준 얼리스탑 +
최적 가중치 복원. 앙상블이면 테스트 예측을 평균합니다.

In [ ]:
criterion = nn.MSELoss()

def train_one(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = SubwayNet(len(NUM_COLS), CARD, EMB_DIMS, HIDDEN_LAYERS, DROPOUT, ACT_MAP[ACTIVATION]).to(device)
    opt = OPT_MAP[OPTIMIZER](model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loader = DataLoader(TensorDataset(Xn_tr, Xc_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    best, best_state, wait = float("inf"), None, 0
    hist_tr, hist_va = [], []
    for epoch in range(EPOCHS):
        model.train(); run = 0.0
        for xn, xc, yb in loader:
            opt.zero_grad(); loss = criterion(model(xn, xc), yb); loss.backward(); opt.step()
            run += loss.item() * len(xn)
        model.eval()
        with torch.no_grad():
            vl = criterion(model(Xn_va, Xc_va), y_va).item()
        hist_tr.append(run/len(Xn_tr)); hist_va.append(vl)
        if vl < best - 1e-6:
            best, best_state, wait = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= EARLY_STOP_PATIENCE: break
    model.load_state_dict(best_state)
    return model, best, hist_tr, hist_va

models, hists = [], None
for i in range(N_ENSEMBLE):
    m, bv, ht, hv = train_one(SEED + i)
    models.append(m)
    if hists is None: hists = (ht, hv)   # 첫 모델 학습곡선 보관
    print(f"  model {i+1}/{N_ENSEMBLE}: best val_loss(표준화)={bv:.4f}  (~R2 {1-bv:.3f}), epochs={len(ht)}")
print("학습 완료.")

# 6. 검증 데이터 평가 (R2)

In [ ]:
def predict(models, xn, xc):
    ps = []
    for m in models:
        m.eval()
        with torch.no_grad():
            ps.append(np.expm1(y_scaler.inverse_transform(m(xn, xc).cpu().numpy())))
    return np.mean(ps, axis=0)

val_pred = predict(models, Xn_va, Xc_va)
val_r2  = r2_score(y_va_orig, val_pred)
val_rmse = np.sqrt(mean_squared_error(y_va_orig, val_pred))
print("=" * 46)
print("[검증 데이터 결과]")
print("  RMSE : %.2f" % val_rmse)
print("  R2   : %.4f  (%.2f%%)" % (val_r2, val_r2 * 100))
print("=" * 46)

# 7. 학습 곡선 & 예측 산점도

In [ ]:
ht, hv = hists
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(ht, label="Train Loss"); ax1.plot(hv, label="Val Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("MSE (log+표준화 y)")
ax1.set_title("Learning Curve (model 1)"); ax1.legend()
ax2.scatter(y_va_orig.flatten(), val_pred.flatten(), alpha=0.4, s=12)
lim = max(y_va_orig.max(), val_pred.max()) * 1.05
ax2.plot([0, lim], [0, lim], "r--", label="Perfect Fit")
ax2.set_xlabel("Actual"); ax2.set_ylabel("Predicted")
ax2.set_title("Val: Actual vs Predicted (R2=%.4f)" % val_r2); ax2.legend()
plt.tight_layout(); plt.show()

# 8. 테스트 데이터 예측 및 저장 (tuned-1의 크래시 셀 수정)

In [ ]:
test_pred = predict(models, Tn, Tc)
test_r2  = r2_score(y_test_true, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test_true, test_pred))
print("=" * 46)
print("[테스트 데이터 최종 결과]")
print("  RMSE : %.2f" % test_rmse)
print("  R2   : %.4f  (%.2f%%)" % (test_r2, test_r2 * 100))
print("=" * 46)
print("※ 검증 R2 ~= 0.87~0.88, 테스트 R2 ~= 0.85 가 이 데이터의 정직한 천장입니다.")

submission = pd.DataFrame({
    "date": test_df["date"],
    "num_people_actual": y_test_true,
    "num_people_predicted": test_pred.flatten(),
})
submission.to_csv("subway_submission_nn.csv", index=False)
print("\nsubway_submission_nn.csv 저장 완료")
print(submission.head(10))